# Notebook 1 — Aquisição e Pré-processamento do Sinal

Primeiro estágio do pipeline de predição de crises epilépticas. Executa três etapas:

1. **Download** dos arquivos EDF dos quatro datasets, incluindo gravações com crise e, quando disponíveis, gravações sem crise (fonte de interictal).
2. **Pré-processamento**: seleção de canais de EEG, filtragem (passa-banda 0.5–40 Hz, notch da rede elétrica) e reamostragem para 256 Hz.
3. **Persistência**: sinal pré-processado e anotações de crise gravados em arquivos `.npz`.

A rotulagem das classes (interictal, pré-ictal, ictal, pós-ictal) é realizada no Notebook 2, após a estimação empírica da janela pré-ictal.

## Posição no pipeline

```
NB1  download + pré-processamento        -> data/signals/*_signal.npz
NB2  estimação de PRE_SEC + rotulagem     -> data/level1_signals/*_L1.npz
NB3  janelamento + extração de features   -> data/features/
NB4  treino e avaliação (A/B/C)           -> data/results/
```

## Estratégia de obtenção de interictal por dataset

A análise dos metadados mostrou que apenas dois datasets possuem gravações inteiramente livres de crise:

| Dataset  | EDFs com crise | EDFs sem crise | Fonte de interictal |
|----------|----------------|----------------|---------------------|
| CHB-MIT  | 75             | 286 (~481 h)   | gravações sem crise |
| SeizeIT2 | 26             | 100 (~913 h)   | gravações sem crise |
| Siena    | 37             | 0              | intervalo intra-arquivo (pré e pós-crise) |
| Mendeley | 20             | 0              | intervalo intra-arquivo (pré e pós-crise) |

Como Siena e Mendeley não disponibilizam gravações sem crise, o interictal é extraído de regiões distantes das crises dentro dos próprios arquivos (Notebook 2). Os arquivos desses datasets são longos (2–14 h), o que permite essa extração.


## 1. Dependências

In [1]:
# Limita o paralelismo do joblib/loky para evitar instabilidade em alguns ambientes
import os
os.environ['LOKY_MAX_CPU_COUNT'] = '1'

%pip install -q numpy pandas matplotlib seaborn mne PyWavelets boto3 scipy tqdm cloudscraper openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re, io, gc, time, tempfile
import html.parser
import urllib.request
import math
from math import gcd

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt, resample_poly
from tqdm.auto import tqdm
import IPython.display as ipd

try:
    import cloudscraper
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'cloudscraper', '-q'])
    import cloudscraper

mne.set_log_level('WARNING')
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('Bibliotecas carregadas.')

c:\Users\danil\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Bibliotecas carregadas.


c:\Users\danil\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## 2. Configuração

Parâmetros centralizados de filtragem, reamostragem e seleção de pacientes.

**Filtragem.** Passa-alta de 0.5 Hz remove deriva de linha de base; passa-baixa de 40 Hz remove ruído de alta frequência e atenua a banda da rede elétrica. Filtro Butterworth de ordem 4 aplicado em modo *zero-phase* (`sosfiltfilt`), sem distorção de fase. Notch na frequência da rede (60 Hz para CHB-MIT, 50 Hz para os demais). Reamostragem para 256 Hz padroniza a taxa entre datasets, condição necessária para comparar features espectrais.


In [3]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Diretórios
ROOT_DIR     = 'data'
CHBMIT_DIR   = os.path.join(ROOT_DIR, 'chb-mit')
SIENA_DIR    = os.path.join(ROOT_DIR, 'siena')
SEIZEIT_DIR  = os.path.join(ROOT_DIR, 'seizeit2')
MENDELEY_DIR = os.path.join(ROOT_DIR, 'mendeley')
SIGNAL_DIR   = os.path.join(ROOT_DIR, 'signals')
L1_DIR       = os.path.join(ROOT_DIR, 'level1_signals')
FEAT_DIR     = os.path.join(ROOT_DIR, 'features')
RESULTS_DIR  = os.path.join(ROOT_DIR, 'results')
for d in [CHBMIT_DIR, SIENA_DIR, SEIZEIT_DIR, MENDELEY_DIR, SIGNAL_DIR, L1_DIR,
          FEAT_DIR, RESULTS_DIR, os.path.join(MENDELEY_DIR, 'Raw_EDF_Files')]:
    os.makedirs(d, exist_ok=True)

# Filtragem e reamostragem
F_HP, F_LP, F_ORDER = 0.5, 40.0, 4
TARGET_FS = 256
NOTCH = {'CHBMIT': 60.0, 'Siena': 50.0, 'SeizeIT2': 50.0, 'Mendeley': 50.0}
SEIZEIT_SESSION = 'ses-01'

# Pacientes selecionados (12 por dataset; Mendeley possui 6 no total)
# PATIENTS = {
#     'CHBMIT'  : ['chb01','chb03','chb04','chb05','chb06','chb07',
#                  'chb08','chb10','chb11','chb12','chb13','chb14'],
#     'Siena'   : ['PN01','PN03','PN05','PN06','PN07',
#                  'PN09','PN10','PN11','PN12','PN13','PN14','PN16'],
#     'Mendeley': ['p10','p11','p12','p13','p14','p15'],
#     'SeizeIT2': ['sub-001','sub-002','sub-003','sub-004','sub-005','sub-006',
#                  'sub-007','sub-008','sub-009','sub-010','sub-011','sub-012'],
# }

PATIENTS = {
    'CHBMIT': [
        'chb01', 'chb03', 'chb04', 'chb05', 'chb06', 'chb07',
        'chb08', 'chb10', 'chb11', 'chb12', 'chb13', 'chb14',
    ],  # 12 — sem mudança
    'Siena': [
        'PN00', 'PN01', 'PN03', 'PN05', 'PN06', 'PN09',
        'PN10', 'PN12', 'PN13', 'PN14', 'PN16', 'PN17',
    ],  # 12 — substitui PN07→PN17, PN11→PN00
    'Mendeley': [
        'p10', 'p11', 'p12', 'p13', 'p14', 'p15',
    ],  # 6 — sem mudança
    'SeizeIT2': [
        'sub-001', 'sub-002', 'sub-004', 'sub-005',
        'sub-007', 'sub-008', 'sub-011', 'sub-012',
        'sub-035', 'sub-039', 'sub-047', 'sub-073',
    ],  # 12 — substitui sub-003/006/009/010 pelos 4 com mais crises
}

# Arquivos excluídos por anomalia confirmada de canal (ver Notebook 1.5 —
# auditoria de canais). chb12_28/29 têm o eletrodo occipital grafado como
# '01' (zero) em vez de 'O1' (letra) no cabeçalho EDF original — bug do
# próprio arquivo fonte, não do pipeline. Occipital cai para 1 canal nesses
# dois arquivos; excluídos aqui para que nunca entrem em nenhuma etapa
# posterior (NB2, NB3), sem precisar excluir o paciente inteiro.
EXCLUDED_FILES = {
    'CHBMIT': {'chb12': ['chb12_28.edf', 'chb12_29.edf']},
}

# Datasets dos quais se baixam gravações sem crise (fonte direta de interictal)
DOWNLOAD_SEIZURE_FREE = {'CHBMIT': True, 'SeizeIT2': True,
                         'Siena': False, 'Mendeley': False}
# Limite de gravações sem crise por paciente (controla volume em disco)
MAX_FREE_EDF_PER_PATIENT = {'CHBMIT': 8, 'SeizeIT2': 6}

# Rótulos de canal aceitos como EEG de escalo (sistema 10-20) após normalização.
# Canais fora desta lista (ECG, EMG, marcadores) são descartados no pré-processamento.
EEG_10_20 = {
    'fp1','fp2','f7','f3','fz','f4','f8','af3','af4',
    't3','t4','t5','t6','t7','t8','p7','p8','ft9','ft10','tp7','tp8','tp9','tp10',
    'c3','cz','c4','fc1','fc2','cp1','cp2','fc5','fc6','cp5','cp6',
    'p3','pz','p4','po3','po4','po7','po8',
    'o1','o2','oz',
}
# Mapeamento eletrodo -> região cortical
ELECTRODE_REGION = {}
for e in ['fp1','fp2','f7','f3','fz','f4','f8','af3','af4']: ELECTRODE_REGION[e]='frontal'
for e in ['t3','t4','t5','t6','t7','t8','p7','p8','ft9','ft10','tp7','tp8']: ELECTRODE_REGION[e]='temporal'
for e in ['c3','cz','c4','fc1','fc2','cp1','cp2']: ELECTRODE_REGION[e]='central'
for e in ['p3','pz','p4','po3','po4']: ELECTRODE_REGION[e]='parietal'
for e in ['o1','o2','oz']: ELECTRODE_REGION[e]='occipital'

print('Configuração carregada.')
print(f'Filtragem: passa-banda {F_HP}-{F_LP} Hz (ordem {F_ORDER}), notch por dataset, reamostragem para {TARGET_FS} Hz.')
for k, v in PATIENTS.items():
    fonte = 'gravações sem crise' if DOWNLOAD_SEIZURE_FREE.get(k) else 'intervalo intra-arquivo (NB2)'
    print(f'  {k:9s}: {len(v)} pacientes | interictal via {fonte}')

Configuração carregada.
Filtragem: passa-banda 0.5-40.0 Hz (ordem 4), notch por dataset, reamostragem para 256 Hz.
  CHBMIT   : 12 pacientes | interictal via gravações sem crise
  Siena    : 12 pacientes | interictal via intervalo intra-arquivo (NB2)
  Mendeley : 6 pacientes | interictal via intervalo intra-arquivo (NB2)
  SeizeIT2 : 12 pacientes | interictal via gravações sem crise


## 3. Plano de dados por paciente

Tabela com o orçamento estimado de sinal pré-ictal e interictal por paciente, calculado a partir dos metadados. Documenta o dimensionamento das classes; o cálculo definitivo em número de janelas é feito no Notebook 3. Os valores assumem janela pré-ictal de 30 min como limite superior.


In [4]:
PLANO = [
    ('CHB-MIT','chb01',42,7,'sem_crise',7,164,2034),
    ('CHB-MIT','chb03',38,7,'sem_crise',7,144,1860),
    ('CHB-MIT','chb04',42,3,'sem_crise',4,118,8725),
    ('CHB-MIT','chb05',39,5,'sem_crise',5,115,2040),
    ('CHB-MIT','chb06',18,7,'sem_crise',10,253,2450),
    ('CHB-MIT','chb07',19,3,'sem_crise',3,90,3481),
    ('CHB-MIT','chb08',20,5,'sem_crise',5,150,900),
    ('CHB-MIT','chb10',25,7,'sem_crise',7,203,2160),
    ('CHB-MIT','chb11',35,3,'sem_crise',3,54,1920),
    ('CHB-MIT','chb12',24,13,'sem_crise',40,362,660),
    ('CHB-MIT','chb13',33,8,'sem_crise',12,202,1500),
    ('CHB-MIT','chb14',26,7,'sem_crise',8,214,1140),
    ('Siena','PN01',1,1,'intra_arquivo',2,60,135),
    ('Siena','PN03',2,2,'intra_arquivo',2,60,1199),
    ('Siena','PN05',3,3,'intra_arquivo',3,90,188),
    ('Siena','PN06',5,5,'intra_arquivo',5,150,349),
    ('Siena','PN07',1,1,'intra_arquivo',1,30,398),
    ('Siena','PN09',3,3,'intra_arquivo',3,90,255),
    ('Siena','PN10',6,6,'intra_arquivo',10,300,439),
    ('Siena','PN11',1,1,'intra_arquivo',1,30,91),
    ('Siena','PN12',3,3,'intra_arquivo',3,65,179),
    ('Siena','PN13',3,3,'intra_arquivo',3,90,259),
    ('Siena','PN14',4,4,'intra_arquivo',4,120,1048),
    ('Siena','PN16',2,2,'intra_arquivo',2,60,194),
    ('Mendeley','p10',2,2,'intra_arquivo',2,60,170),
    ('Mendeley','p11',4,4,'intra_arquivo',7,205,191),
    ('Mendeley','p12',3,3,'intra_arquivo',7,173,146),
    ('Mendeley','p13',4,4,'intra_arquivo',6,174,190),
    ('Mendeley','p14',3,3,'intra_arquivo',8,235,40),
    ('Mendeley','p15',4,4,'intra_arquivo',5,150,258),
    ('SeizeIT2','sub-001',9,4,'sem_crise',4,120,2920),
    ('SeizeIT2','sub-002',12,7,'sem_crise',15,430,1481),
    ('SeizeIT2','sub-003',10,1,'sem_crise',1,30,5610),
    ('SeizeIT2','sub-004',1,1,'sem_crise',4,113,0),
    ('SeizeIT2','sub-005',26,2,'sem_crise',2,60,6555),
    ('SeizeIT2','sub-006',10,1,'sem_crise',1,30,6137),
    ('SeizeIT2','sub-007',5,1,'sem_crise',2,60,2923),
    ('SeizeIT2','sub-008',11,2,'sem_crise',2,60,6600),
    ('SeizeIT2','sub-009',10,1,'sem_crise',1,30,5755),
    ('SeizeIT2','sub-010',11,1,'sem_crise',1,30,5898),
    ('SeizeIT2','sub-011',11,3,'sem_crise',4,90,5714),
    ('SeizeIT2','sub-012',10,2,'sem_crise',3,90,5212),
]
df_plano = pd.DataFrame(PLANO, columns=[
    'dataset','paciente','n_edf','n_edf_crise','fonte_interictal',
    'crises','preictal_min','interictal_disp_min'])
df_plano['interictal_1a1_min'] = df_plano[['preictal_min','interictal_disp_min']].min(axis=1)

print('Plano por paciente:')
ipd.display(df_plano)

resumo = df_plano.groupby('dataset').agg(
    pacientes=('paciente','count'), crises=('crises','sum'),
    preictal_h=('preictal_min', lambda x: round(x.sum()/60,1)),
    interictal_disp_h=('interictal_disp_min', lambda x: round(x.sum()/60,1)),
    treino_1a1_h=('interictal_1a1_min', lambda x: round(x.sum()/60,1)),
).reset_index()
print('\nResumo por dataset (horas):')
ipd.display(resumo)
df_plano.to_csv(os.path.join(SIGNAL_DIR, 'plano_dados.csv'), index=False)

Plano por paciente:


,dataset,paciente,n_edf,n_edf_crise,fonte_interictal,crises,preictal_min,interictal_disp_min,interictal_1a1_min
0,CHB-MIT,chb01,42,7,sem_crise,7,164,2034,164
1,CHB-MIT,chb03,38,7,sem_crise,7,144,1860,144
2,CHB-MIT,chb04,42,3,sem_crise,4,118,8725,118
3,CHB-MIT,chb05,39,5,sem_crise,5,115,2040,115
4,CHB-MIT,chb06,18,7,sem_crise,10,253,2450,253
5,CHB-MIT,chb07,19,3,sem_crise,3,90,3481,90
6,CHB-MIT,chb08,20,5,sem_crise,5,150,900,150
7,CHB-MIT,chb10,25,7,sem_crise,7,203,2160,203
8,CHB-MIT,chb11,35,3,sem_crise,3,54,1920,54
9,CHB-MIT,chb12,24,13,sem_crise,40,362,660,362



Resumo por dataset (horas):


,dataset,pacientes,crises,preictal_h,interictal_disp_h,treino_1a1_h
0,CHB-MIT,12,111,34.5,481.2,34.5
1,Mendeley,6,35,16.6,16.6,12.7
2,SeizeIT2,12,40,19.0,913.4,17.2
3,Siena,12,39,19.1,78.9,19.1


## 4. Seleção de canais e filtragem

`select_eeg_channels` mantém apenas eletrodos do sistema 10-20, descartando canais não-EEG (ECG, EMG, marcadores fotônicos). Para o SeizeIT2, cujos eletrodos *behind-the-ear* não seguem a nomenclatura padrão, todos os canais EEG presentes são mantidos e tratados como temporais nas etapas seguintes.

`apply_filters` aplica o passa-banda e o notch. A frequência de corte superior é limitada abaixo da frequência de Nyquist para evitar coeficientes inválidos caso algum arquivo tenha taxa de amostragem baixa.


In [5]:
def normalize_electrode(name):
    s = str(name).lower()
    for token in ['eeg', '-ref', '-le', '-a1', '-a2', 'ref']:
        s = s.replace(token, ' ')
    s = s.strip()
    if '-' in s:
        s = s.split('-')[0]
    return re.sub(r'[^a-z0-9]', '', s)

def channel_region(name, dataset=None):
    if dataset == 'SeizeIT2':
        return 'temporal'
    return ELECTRODE_REGION.get(normalize_electrode(name), None)

def select_eeg_channels(ch_names, dataset):
    if dataset == 'SeizeIT2':
        return list(range(len(ch_names))), [str(c) for c in ch_names]
    keep_idx, keep_names = [], []
    for i, c in enumerate(ch_names):
        if normalize_electrode(c) in EEG_10_20:
            keep_idx.append(i)
            keep_names.append(str(c))
    return keep_idx, keep_names

def _bandpass_sos(fs):
    ny = fs / 2.0
    return butter(F_ORDER, [max(F_HP, 0.1)/ny, min(F_LP, ny*0.99)/ny],
                  btype='band', output='sos')

def _notch_ba(fs, notch_freq):
    if notch_freq < fs / 2:
        return iirnotch(notch_freq, Q=30, fs=fs)
    return None, None

CHUNK_MIN = 20

def process_edf(edf_path, dataset, patient, fkey, iv, has_seizure):
    # ── 1. Cabeçalho (sem carregar sinal) ──
    raw = mne.io.read_raw_edf(str(edf_path), preload=False, verbose=False)
    fs        = raw.info['sfreq']
    all_names = raw.ch_names
    n_total   = raw.n_times
    raw.close()

    keep_idx, keep_names = select_eeg_channels(all_names, dataset)
    if not keep_idx:
        raise ValueError('nenhum canal EEG reconhecido')

    n_ch     = len(keep_idx)
    notch    = NOTCH.get(dataset, 50.0)
    sos      = _bandpass_sos(fs)
    b_n, a_n = _notch_ba(fs, notch)
    too_short = n_total < 3 * (F_ORDER * 2 + 1)

    # Sanitiza os intervalos de crise em relação à duração real do arquivo.
    # Descarta crises cujo onset >= duração (anotação de outro arquivo ou erro do xlsx).
    # Clampeia offsets que ultrapassam o fim por menos de 60 s (erro de arredondamento).
    dur_s = n_total / fs
    iv_clean = []
    for o, e in iv:
        if o >= dur_s:
            continue          # onset fora do arquivo — descarta
        e = min(e, dur_s)     # offset além do fim — clampeia
        if e > o:
            iv_clean.append((o, e))
    iv = iv_clean

    g    = gcd(int(round(fs)), TARGET_FS)
    up   = TARGET_FS // g
    down = int(round(fs)) // g
    need_resample = int(round(fs)) != TARGET_FS

    chunk_samp = int(CHUNK_MIN * 60 * fs)
    n_chunks   = math.ceil(n_total / chunk_samp)

    if need_resample:
        n_out = math.ceil(n_total * up / down)
    else:
        n_out = n_total

    # ── 2. Memmap em disco ──
    tmp_path = str(edf_path) + '.tmp.dat'
    try:
        mmap = np.memmap(tmp_path, dtype=np.float32, mode='w+',
                         shape=(n_ch, n_out))
        out_col = 0

        for chunk_i in range(n_chunks):
            s_start = chunk_i * chunk_samp
            s_end   = min(s_start + chunk_samp, n_total)

            raw = mne.io.read_raw_edf(str(edf_path), preload=False, verbose=False)
            t_start = s_start / fs
            t_end   = min(s_end / fs, (n_total - 1) / fs)  # clampado ao último sample válido
            raw.crop(tmin=t_start, tmax=t_end)
            raw.load_data(verbose=False)

            chunk = raw.get_data(picks=keep_idx) * 1e6
            raw.close(); del raw

            chunk = chunk.astype(np.float64)

            if not too_short and chunk.shape[1] >= 3 * (F_ORDER * 2 + 1):
                chunk = sosfiltfilt(sos, chunk, axis=1)
                if b_n is not None:
                    chunk = filtfilt(b_n, a_n, chunk, axis=1)

            if need_resample:
                chunk = resample_poly(chunk, up, down, axis=1)

            n_col   = chunk.shape[1]
            end_col = min(out_col + n_col, n_out)
            actual  = end_col - out_col
            mmap[:, out_col:end_col] = chunk[:, :actual].astype(np.float32)
            out_col = end_col
            del chunk; gc.collect()

        mmap.flush()

        # ── 3. Grava .npz ──
        out_path = os.path.join(SIGNAL_DIR,
                                f'{dataset}__{patient}__{fkey}_signal.npz')
        np.savez_compressed(
            out_path,
            data=mmap,
            sfreq=np.float32(TARGET_FS if need_resample else fs),
            ch_names=np.array(keep_names),
            dataset=np.str_(dataset), patient=np.str_(patient),
            seizure_intervals=(np.array(iv, dtype=np.float32).reshape(-1, 2)
                               if iv else np.empty((0, 2), dtype=np.float32)),
            has_seizure=np.bool_(has_seizure),
        )
        del mmap

    finally:
        try:
            del mmap
        except NameError:
            pass
        gc.collect()
        if os.path.exists(tmp_path):
            try:
                os.remove(tmp_path)
            except OSError:
                pass

    gc.collect()
    return out_path

print('Funções de filtragem definidas (MNE chunk de 20 min + memmap, pico ~120 MB).')

Funções de filtragem definidas (MNE chunk de 20 min + memmap, pico ~120 MB).


## 5. Download — CHB-MIT

Acesso anônimo ao bucket público da PhysioNet no S3. O arquivo `summary.txt` de cada paciente indica, por gravação, o número de crises e seus instantes de início e fim. Gravações com crise são todas baixadas; gravações sem crise são baixadas até o limite definido por paciente.


In [6]:
def s3_client():
    return boto3.client('s3', region_name='us-east-1',
                        config=Config(signature_version=UNSIGNED))

CHBMIT_BUCKET = 'physionet-open'
CHBMIT_PREFIX = 'chbmit/1.0.0/'

def _parse_chbmit_summary(sum_path):
    '''Lê o summary.txt e retorna ({arquivo: [(onset_s, offset_s)]}, [arquivos sem crise]).'''
    seizure_edfs, free_edfs = {}, []
    with open(sum_path, encoding='utf-8', errors='ignore') as f:
        blocks = f.read().split('File Name:')
    for block in blocks[1:]:
        lines = block.strip().split('\n')
        fname = lines[0].strip()
        n_sz = 0; onsets = []; offsets = []
        for ln in lines:
            if ln.startswith('Number of Seizures in File:'):
                try: n_sz = int(ln.split(':')[1].strip())
                except ValueError: pass
            elif 'Seizure' in ln and 'Start Time' in ln:
                m = re.search(r'(\d+)', ln.split(':', 1)[1]); onsets.append(int(m.group(1)) if m else None)
            elif 'Seizure' in ln and 'End Time' in ln:
                m = re.search(r'(\d+)', ln.split(':', 1)[1]); offsets.append(int(m.group(1)) if m else None)
        if n_sz > 0:
            iv = [(o, e) for o, e in zip(onsets, offsets) if o is not None and e is not None and e > o]
            seizure_edfs[fname] = iv
        else:
            free_edfs.append(fname)
    return seizure_edfs, free_edfs

def download_chbmit_patient(patient, max_free=None):
    s3 = s3_client()
    pat_dir = os.path.join(CHBMIT_DIR, patient); os.makedirs(pat_dir, exist_ok=True)
    sum_local = os.path.join(pat_dir, f'{patient}-summary.txt')
    if not os.path.exists(sum_local):
        s3.download_file(CHBMIT_BUCKET, f'{CHBMIT_PREFIX}{patient}/{patient}-summary.txt', sum_local)
    seizure_edfs, free_edfs = _parse_chbmit_summary(sum_local)

    max_free = max_free if max_free is not None else MAX_FREE_EDF_PER_PATIENT.get('CHBMIT')
    if max_free is not None:
        free_edfs = free_edfs[:max_free]

    targets = list(seizure_edfs.keys()) + (free_edfs if DOWNLOAD_SEIZURE_FREE['CHBMIT'] else [])
    dl = 0
    for fname in tqdm(targets, desc=f'CHB-MIT {patient}', leave=False):
        local = os.path.join(pat_dir, fname)
        if not os.path.exists(local):
            try:
                s3.download_file(CHBMIT_BUCKET, CHBMIT_PREFIX + patient + '/' + fname, local); dl += 1
            except Exception as e:
                print(f'  {fname}: {e}')
    print(f'CHB-MIT {patient}: {len(seizure_edfs)} com crise + {len(free_edfs)} sem crise | {dl} novos')
    return seizure_edfs, free_edfs

for _p in PATIENTS['CHBMIT']:
    download_chbmit_patient(_p)

CHB-MIT chb01: 7 com crise + 8 sem crise | 0 novos


CHB-MIT chb03: 7 com crise + 8 sem crise | 0 novos


CHB-MIT chb04: 3 com crise + 8 sem crise | 0 novos


CHB-MIT chb05: 5 com crise + 8 sem crise | 0 novos


CHB-MIT chb06: 7 com crise + 8 sem crise | 0 novos


...

CHB-MIT chb10: 7 com crise + 8 sem crise | 0 novos


CHB-MIT chb11: 3 com crise + 8 sem crise | 0 novos


CHB-MIT chb12: 13 com crise + 8 sem crise | 0 novos


CHB-MIT chb13: 8 com crise + 8 sem crise | 0 novos


CHB-MIT chb14: 7 com crise + 8 sem crise | 0 novos


## 6. Download — Siena

Acesso por HTTPS ao repositório público da PhysioNet. O dataset não possui gravações sem crise. Os arquivos com crise são identificados pelo `Seizures-list` de cada paciente; o interictal correspondente é extraído por intervalo intra-arquivo no Notebook 2.


In [7]:
SIENA_BASE = 'https://physionet.org/files/siena-scalp-eeg/1.0.0/'

class _LinkParser(html.parser.HTMLParser):
    '''Extrai hrefs de uma listagem de diretório HTML.'''
    def __init__(self):
        super().__init__(); self.links = []
    def handle_starttag(self, tag, attrs):
        if tag == 'a':
            for a, v in attrs:
                if a == 'href' and not v.startswith(('?', '..')):
                    self.links.append(v)

def _edf_prefix(fname):
    '''Prefixo do paciente a partir do nome do arquivo (normaliza pno6 -> pn06).'''
    prefix = re.sub(r'[-_].*', '', fname.lower().replace('.edf', ''))
    prefix = re.sub(r'o(\d)', r'0\1', prefix)
    return prefix

def _download_with_progress(url, local_path, desc=''):
    tmp = local_path + '.part'
    try:
        with urllib.request.urlopen(url, timeout=60) as r:
            total = int(r.headers.get('Content-Length', 0)); done = 0
            with open(tmp, 'wb') as f:
                while True:
                    chunk = r.read(1024 * 64)
                    if not chunk:
                        break
                    f.write(chunk); done += len(chunk)
                    pct = done / total * 100 if total else 0
                    bar = int(pct / 2)
                    print(f'\r  {desc}: {pct:5.1f}% |{"#" * bar}{" " * (50 - bar)}| '
                          f'{done/1e6:.1f}/{total/1e6:.1f} MB', end='', flush=True)
            print()
        os.replace(tmp, local_path); return True
    except Exception:
        print()
        if os.path.exists(tmp):
            os.remove(tmp)
        raise

def download_siena_patient(patient):
    pat_dir = os.path.join(SIENA_DIR, patient); os.makedirs(pat_dir, exist_ok=True)
    base = SIENA_BASE + patient + '/'
    try:
        with urllib.request.urlopen(base, timeout=30) as r:
            content = r.read().decode('utf-8')
    except Exception as e:
        print(f'  {patient}: {e}'); return
    parser = _LinkParser(); parser.feed(content)
    all_files = [l.rstrip('/').split('/')[-1] for l in parser.links if l.rstrip('/').split('/')[-1]]
    # anotações (.txt)
    for fn in [l for l in all_files if l.lower().endswith('.txt')]:
        local = os.path.join(pat_dir, fn)
        if not os.path.exists(local):
            try: urllib.request.urlretrieve(base + fn, local)
            except Exception as e: print(f'  {fn}: {e}')
    # arquivos com crise listados no Seizures-list
    seizure_edfs = set()
    sl = os.path.join(pat_dir, f'Seizures-list-{patient}.txt')
    if os.path.exists(sl):
        with open(sl, encoding='utf-8', errors='ignore') as f:
            for line in f:
                m = re.search(r'File name:\s*(\S+\.edf)', line, re.IGNORECASE)
                if m:
                    seizure_edfs.add(re.sub(r'-+(\.edf)$', r'\1', m.group(1).strip()).lower())
    edf_files = [l for l in all_files if l.lower().endswith('.edf')]
    if seizure_edfs:
        sz_pref = {_edf_prefix(e) for e in seizure_edfs}
        targets = [e for e in edf_files if _edf_prefix(e) in sz_pref]
    else:
        targets = edf_files
    if not targets:
        targets = edf_files
    dl = 0
    for i, fn in enumerate(targets, 1):
        local = os.path.join(pat_dir, fn)
        if not os.path.exists(local):
            try:
                _download_with_progress(base + fn, local, f'Siena {patient} [{i}/{len(targets)}] {fn}'); dl += 1
            except Exception as e:
                print(f'  {fn}: {e}')
    print(f'Siena {patient}: {len(targets)} arquivos com crise | {dl} novos')

for _p in PATIENTS['Siena']:
    download_siena_patient(_p)

Siena PN00: 5 arquivos com crise | 0 novos
Siena PN01: 1 arquivos com crise | 0 novos
Siena PN03: 2 arquivos com crise | 0 novos
Siena PN05: 3 arquivos com crise | 0 novos
Siena PN06: 5 arquivos com crise | 0 novos
Siena PN09: 3 arquivos com crise | 0 novos
Siena PN10: 6 arquivos com crise | 0 novos
Siena PN12: 3 arquivos com crise | 0 novos
Siena PN13: 3 arquivos com crise | 0 novos
Siena PN14: 4 arquivos com crise | 0 novos
Siena PN16: 2 arquivos com crise | 0 novos
Siena PN17: 2 arquivos com crise | 0 novos


## 7. Download — SeizeIT2

Acesso anônimo ao bucket do OpenNeuro (`ds005873`). Os arquivos de eventos (`_events.tsv`), de baixo volume, são baixados primeiro para identificar quais gravações contêm crise. Gravações com crise são todas baixadas; gravações sem crise, até o limite por paciente.


In [8]:
SEIZEIT_BUCKET = 'openneuro.org'
SEIZEIT_DATASET = 'ds005873'

def _tsv_has_seizure(tsv_bytes):
    try:
        df = pd.read_csv(io.BytesIO(tsv_bytes), sep='\t')
    except Exception:
        return False
    if 'eventType' not in df.columns:
        return False
    et = df['eventType'].astype(str).str.lower().str.strip()
    return bool(et.str.startswith('sz').any())

def download_seizeit2_subject(subject, max_free=None):
    s3 = s3_client()
    prefix = f'{SEIZEIT_DATASET}/{subject}/{SEIZEIT_SESSION}/eeg/'
    sub_dir = os.path.join(SEIZEIT_DIR, subject, SEIZEIT_SESSION, 'eeg'); os.makedirs(sub_dir, exist_ok=True)
    paginator = s3.get_paginator('list_objects_v2'); keys = []
    for page in paginator.paginate(Bucket=SEIZEIT_BUCKET, Prefix=prefix):
        keys.extend(o['Key'] for o in page.get('Contents', []))
    seizure_runs = set(); all_runs = set()
    for key in [k for k in keys if k.endswith('_events.tsv')]:
        fn = key.split('/')[-1]; local = os.path.join(sub_dir, fn)
        run = fn.replace('_events.tsv', ''); all_runs.add(run)
        if not os.path.exists(local):
            try: s3.download_file(SEIZEIT_BUCKET, key, local)
            except Exception as e: print(f'  {fn}: {e}')
        try:
            with open(local, 'rb') as f:
                if _tsv_has_seizure(f.read()):
                    seizure_runs.add(run)
        except Exception:
            pass
    free_runs = sorted(all_runs - seizure_runs)
    max_free = max_free if max_free is not None else MAX_FREE_EDF_PER_PATIENT.get('SeizeIT2')
    if not DOWNLOAD_SEIZURE_FREE['SeizeIT2']:
        free_runs = []
    elif max_free is not None:
        free_runs = free_runs[:max_free]
    keep_runs = seizure_runs | set(free_runs)
    dl = 0
    for key in tqdm([k for k in keys if k.endswith(('_eeg.edf', '_eeg.json'))],
                    desc=f'SeizeIT2 {subject}', leave=False):
        fn = key.split('/')[-1]; run = fn.replace('_eeg.edf', '').replace('_eeg.json', '')
        if run not in keep_runs:
            continue
        local = os.path.join(sub_dir, fn)
        if not os.path.exists(local):
            try: s3.download_file(SEIZEIT_BUCKET, key, local); dl += 1
            except Exception as e: print(f'  {fn}: {e}')
    print(f'SeizeIT2 {subject}: {len(seizure_runs)} com crise + {len(free_runs)} sem crise | {dl} novos')

for _p in PATIENTS['SeizeIT2']:
    download_seizeit2_subject(_p)

SeizeIT2 sub-001: 4 com crise + 5 sem crise | 0 novos


SeizeIT2 sub-002: 7 com crise + 5 sem crise | 0 novos


SeizeIT2 sub-004: 1 com crise + 0 sem crise | 0 novos


SeizeIT2 sub-005: 2 com crise + 6 sem crise | 0 novos


SeizeIT2 sub-007: 1 com crise + 4 sem crise | 0 novos


...

SeizeIT2 sub-012: 2 com crise + 6 sem crise | 0 novos


SeizeIT2 sub-035: 8 com crise + 6 sem crise | 14 novos


SeizeIT2 sub-039: 7 com crise + 2 sem crise | 9 novos


SeizeIT2 sub-047: 6 com crise + 1 sem crise | 7 novos


SeizeIT2 sub-073: 26 com crise + 6 sem crise | 32 novos


## 8. Download — Mendeley

Acesso pela API pública do Mendeley Data (`cloudscraper` contorna a proteção Cloudflare). O dataset não possui gravações sem crise; todos os arquivos dos seis pacientes são baixados, junto da planilha de anotações usada no Notebook 2.


In [9]:
MEND_DATASET_ID = '5pc2j46cbc'
MEND_VERSION = 1
MEND_EDF_FOLDER = '3bc6cc31-0156-490b-8e39-4b740598b289'
MEND_DOC_FOLDER = '465c0896-7d08-4fbe-8ee3-378beca656d5'
_MEND_HEADERS = {'Accept': 'application/vnd.mendeley-public-dataset.1+json'}

def download_mendeley_edfs(patients=None):
    patients = patients if patients is not None else PATIENTS['Mendeley']
    scraper = cloudscraper.create_scraper()
    edf_dir = os.path.join(MENDELEY_DIR, 'Raw_EDF_Files'); os.makedirs(edf_dir, exist_ok=True)
    doc_dir = os.path.join(MENDELEY_DIR, 'Documentation'); os.makedirs(doc_dir, exist_ok=True)
    # planilha de anotações de crise
    url_doc = (f'https://data.mendeley.com/public-api/datasets/{MEND_DATASET_ID}'
               f'/files?folder_id={MEND_DOC_FOLDER}&version={MEND_VERSION}')
    try:
        for fobj in scraper.get(url_doc, headers=_MEND_HEADERS, timeout=30).json():
            if fobj['filename'] == 'Seizures_Information.xlsx':
                dst = os.path.join(doc_dir, 'Seizures_Information.xlsx')
                if not os.path.exists(dst):
                    with open(dst, 'wb') as f:
                        f.write(scraper.get(fobj['content_details']['download_url'], timeout=60).content)
                break
    except Exception as e:
        print(f'  documentação: {e}')
    # arquivos EDF
    url = (f'https://data.mendeley.com/public-api/datasets/{MEND_DATASET_ID}'
           f'/files?folder_id={MEND_EDF_FOLDER}&version={MEND_VERSION}')
    r = scraper.get(url, headers=_MEND_HEADERS, timeout=30); r.raise_for_status()
    def _keep(fn):
        return any(fn.lower().startswith(p.lower() + '_') or fn.lower().startswith(p.lower() + 'record')
                   for p in patients)
    targets = [f for f in r.json() if f['filename'].lower().endswith('.edf') and _keep(f['filename'])]
    dl = 0
    for fobj in tqdm(targets, desc='Mendeley EDFs'):
        dst = os.path.join(edf_dir, fobj['filename'])
        if not os.path.exists(dst):
            with open(dst, 'wb') as f:
                f.write(scraper.get(fobj['content_details']['download_url'], timeout=120).content)
            dl += 1
    print(f'Mendeley: {len(targets)} arquivos selecionados | {dl} novos')

download_mendeley_edfs()

Mendeley EDFs: 100%|██████████| 20/20 [00:00<00:00, 2973.00it/s]

Mendeley: 20 arquivos selecionados | 0 novos


## 9. Leitura de anotações de crise

Funções que extraem os intervalos de crise (em segundos, relativos ao início do arquivo) de cada formato de anotação: `summary.txt` (CHB-MIT), `Seizures-list` (Siena), `_events.tsv` (SeizeIT2) e `Seizures_Information.xlsx` (Mendeley).


In [10]:
def _hms_to_sec(s):
    '''Converte HH.MM.SS (ou HH:MM:SS) em segundos.'''
    s = str(s).strip().replace(':', '.'); p = s.split('.')
    if len(p) >= 3:
        return int(p[0]) * 3600 + int(p[1]) * 60 + int(float(p[2]))
    return 0

def chbmit_intervals(patient):
    sum_local = os.path.join(CHBMIT_DIR, patient, f'{patient}-summary.txt')
    if not os.path.exists(sum_local):
        return {}, []
    return _parse_chbmit_summary(sum_local)

def siena_intervals(patient):
    '''{arquivo: [(onset_s, offset_s)]} a partir do Seizures-list.'''
    sl = os.path.join(SIENA_DIR, patient, f'Seizures-list-{patient}.txt')
    out = {}
    if not os.path.exists(sl):
        return out
    cur = None; reg = None
    with open(sl, encoding='utf-8', errors='ignore') as f:
        for raw in f:
            line = raw.strip()
            m = re.match(r'File name:\s*(\S+\.edf)', line, re.IGNORECASE)
            if m:
                cur = re.sub(r'-+(\.edf)$', r'\1', m.group(1).strip()).lower(); reg = None
                out.setdefault(cur, []); continue
            m = re.match(r'Registration start time:\s*([\d.:]+)', line, re.IGNORECASE)
            if m:
                reg = _hms_to_sec(m.group(1)); continue
            m = re.match(r'(?:Seizure\s+)?Start\s+time:\s*([\d.:]+)', line, re.IGNORECASE)
            if m and cur is not None and reg is not None:
                t = _hms_to_sec(m.group(1)) - reg
                if t < 0: t += 86400
                out[cur].append([t, None]); continue
            m = re.match(r'(?:Seizure\s+)?End\s+time:\s*([\d.:]+)', line, re.IGNORECASE)
            if m and cur is not None and reg is not None:
                t = _hms_to_sec(m.group(1)) - reg
                if t < 0: t += 86400
                if out[cur] and out[cur][-1][1] is None:
                    out[cur][-1][1] = t
    return {k: [(o, e) for o, e in v if e and e > o] for k, v in out.items()}

def seizeit2_intervals(tsv_path):
    try:
        df = pd.read_csv(tsv_path, sep='\t')
    except Exception:
        return []
    if 'eventType' not in df.columns:
        return []
    et = df['eventType'].astype(str).str.lower().str.strip()
    iv = []
    for _, row in df[et.str.startswith('sz')].iterrows():
        try:
            o = float(row['onset']); d = float(row['duration'])
            if d > 0:
                iv.append((o, o + d))
        except (ValueError, KeyError):
            pass
    return iv

def mendeley_intervals():
    '''{arquivo: [(onset_s, offset_s)]} a partir de Seizures_Information.xlsx.'''
    xlsx = os.path.join(MENDELEY_DIR, 'Documentation', 'Seizures_Information.xlsx')
    if not os.path.exists(xlsx):
        return {}
    raw = pd.read_excel(xlsx, header=None); data = raw.iloc[3:].reset_index(drop=True)
    I_FILE, I_FONS, I_SON, I_SDUR = 7, 9, 12, 13
    def dur_to_sec(t):
        t = str(t); mm = re.search(r'(\d+)\s*min', t); ss = re.search(r'(\d+)\s*sec', t)
        return (int(mm.group(1)) * 60 if mm else 0) + (int(ss.group(1)) if ss else 0)
    out = {}; cur = None; fons = None
    for _, row in data.iterrows():
        if pd.notna(row.iloc[I_FILE]):
            fn = str(row.iloc[I_FILE]).strip()
            if not fn.lower().endswith('.edf'):
                fn += '.edf'
            cur = fn; out.setdefault(cur, [])
            fons = _hms_to_sec(row.iloc[I_FONS]) if pd.notna(row.iloc[I_FONS]) else 0
        if cur and pd.notna(row.iloc[I_SON]) and pd.notna(row.iloc[I_SDUR]):
            son = _hms_to_sec(str(row.iloc[I_SON]).strip()); d = dur_to_sec(row.iloc[I_SDUR])
            if d:
                rel = son - (fons or 0)
                if rel < 0: rel += 86400
                out[cur].append((rel, rel + d))
    return out

print('Funções de leitura de anotações definidas.')

Funções de leitura de anotações definidas.


## 10. Pré-processamento e gravação

Cada arquivo EDF baixado é lido, tem seus canais EEG selecionados, é filtrado, reamostrado e gravado em `.npz`. Arquivos já processados são ignorados, permitindo reexecução incremental.


In [11]:
def process_chbmit():
    for pat in PATIENTS['CHBMIT']:
        pat_dir = os.path.join(CHBMIT_DIR, pat)
        if not os.path.isdir(pat_dir):
            continue
        sz_map, _ = _parse_chbmit_summary(os.path.join(pat_dir, f'{pat}-summary.txt'))
        edfs = [f for f in os.listdir(pat_dir) if f.endswith('.edf')]
        excl = EXCLUDED_FILES.get('CHBMIT', {}).get(pat, [])
        if excl:
            edfs = [f for f in edfs if f not in excl]
        for fn in tqdm(edfs, desc=f'CHB-MIT {pat}', leave=False):
            fkey = fn.replace('.edf', '')
            out  = os.path.join(SIGNAL_DIR, f'CHBMIT__{pat}__{fkey}_signal.npz')
            if os.path.exists(out):
                continue
            try:
                process_edf(os.path.join(pat_dir, fn), 'CHBMIT', pat, fkey,
                            sz_map.get(fn, []), has_seizure=len(sz_map.get(fn,[])) > 0)
            except Exception as e:
                print(f'  {fn}: {e}')
    print('CHB-MIT pré-processado.')

process_chbmit()

CHB-MIT pré-processado.


In [12]:
def process_siena():
    for pat in PATIENTS['Siena']:
        pat_dir = os.path.join(SIENA_DIR, pat)
        if not os.path.isdir(pat_dir):
            continue
        iv_map = siena_intervals(pat)
        edfs = [f for f in os.listdir(pat_dir) if f.lower().endswith('.edf')]
        for fn in tqdm(edfs, desc=f'Siena {pat}', leave=False):
            fkey = fn.replace('.edf', '')
            out  = os.path.join(SIGNAL_DIR, f'Siena__{pat}__{fkey}_signal.npz')
            if os.path.exists(out):
                continue
            iv = iv_map.get(fn.lower(), [])
            if not iv:
                for k, v in iv_map.items():
                    if _edf_prefix(k) == _edf_prefix(fn):
                        iv = v; break
            try:
                process_edf(os.path.join(pat_dir, fn), 'Siena', pat, fkey,
                            iv, has_seizure=len(iv) > 0)
            except Exception as e:
                print(f'  {fn}: {e}')
    print('Siena pré-processado.')

process_siena()

Siena pré-processado.


In [13]:
def process_seizeit2():
    for sub in PATIENTS['SeizeIT2']:
        eeg_dir = os.path.join(SEIZEIT_DIR, sub, SEIZEIT_SESSION, 'eeg')
        if not os.path.isdir(eeg_dir):
            continue
        edfs = [f for f in os.listdir(eeg_dir) if f.endswith('_eeg.edf')]
        for fn in tqdm(edfs, desc=f'SeizeIT2 {sub}', leave=False):
            fkey = fn.replace('_eeg.edf', '')
            out  = os.path.join(SIGNAL_DIR, f'SeizeIT2__{sub}__{fkey}_signal.npz')
            if os.path.exists(out):
                continue
            tsv = os.path.join(eeg_dir, fkey + '_events.tsv')
            iv  = seizeit2_intervals(tsv) if os.path.exists(tsv) else []
            try:
                process_edf(os.path.join(eeg_dir, fn), 'SeizeIT2', sub, fkey,
                            iv, has_seizure=len(iv) > 0)
            except Exception as e:
                print(f'  {fn}: {e}')
    print('SeizeIT2 pré-processado.')

process_seizeit2()

SeizeIT2 pré-processado.


In [14]:
def process_mendeley():
    edf_dir = os.path.join(MENDELEY_DIR, 'Raw_EDF_Files')
    if not os.path.isdir(edf_dir):
        print('Sem arquivos EDF do Mendeley.'); return
    iv_map = mendeley_intervals()
    def _pat_of(fn):
        m = re.match(r'(p\d+)', fn.lower()); return m.group(1) if m else None
    edfs = [f for f in os.listdir(edf_dir) if f.lower().endswith('.edf')]
    for fn in tqdm(edfs, desc='Mendeley'):
        pat = _pat_of(fn)
        if pat not in PATIENTS['Mendeley']:
            continue
        fkey = fn.replace('.edf', '')
        out  = os.path.join(SIGNAL_DIR, f'Mendeley__{pat}__{fkey}_signal.npz')
        if os.path.exists(out):
            continue
        iv = iv_map.get(fn, iv_map.get(fkey + '.edf', []))
        try:
            process_edf(os.path.join(edf_dir, fn), 'Mendeley', pat, fkey,
                        iv, has_seizure=len(iv) > 0)
        except Exception as e:
            print(f'  {fn}: {e}')
    print('Mendeley pré-processado.')

process_mendeley()

Mendeley: 100%|██████████| 20/20 [00:00<00:00, 3799.70it/s]

Mendeley pré-processado.


## 11. Verificação dos sinais gravados

Inventário dos arquivos `.npz` produzidos: contagem por dataset, separação entre gravações com e sem crise, duração total e número de crises anotadas.


In [15]:
import glob
sigs = sorted(glob.glob(os.path.join(SIGNAL_DIR, '*_signal.npz')))
rows = []
for f in sigs:
    base = os.path.basename(f)
    m = re.match(r'(.+?)__(.+?)__(.+)_signal\.npz', base)
    if not m:
        continue
    z = np.load(f, allow_pickle=True)
    rows.append(dict(dataset=m.group(1), patient=m.group(2),
                     dur_min=round(z['data'].shape[1] / float(z['sfreq']) / 60, 1),
                     n_ch=z['data'].shape[0],
                     has_seizure=bool(z['has_seizure']),
                     n_seizures=len(z['seizure_intervals'])))
dfx = pd.DataFrame(rows)
print(f'Sinais gravados: {len(dfx)}\n')
if not dfx.empty:
    g = dfx.groupby('dataset').agg(
        sinais=('patient', 'count'),
        com_crise=('has_seizure', 'sum'),
        sem_crise=('has_seizure', lambda x: (~x).sum()),
        horas=('dur_min', lambda x: round(x.sum() / 60, 1)),
        crises=('n_seizures', 'sum')).reset_index()
    ipd.display(g)
    print('\nNúmero de canais EEG por dataset (após seleção):')
    ipd.display(dfx.groupby('dataset')['n_ch'].agg(['min', 'max']))
print('\nPré-processamento concluído. Próximo: Notebook 2.')

Sinais gravados: 382



,dataset,sinais,com_crise,sem_crise,horas,crises
0,CHBMIT,171,74,97,284.8,105
1,Mendeley,20,20,0,57.8,34
2,SeizeIT2,150,73,77,1255.0,309
3,Siena,41,41,0,131.8,43



Número de canais EEG por dataset (após seleção):


,min,max
dataset,,
CHBMIT,18,23
Mendeley,17,19
SeizeIT2,2,2
Siena,27,27



Pré-processamento concluído. Próximo: Notebook 2.


In [16]:
import glob, random

print('=== Verificação de integridade dos _signal.npz ===\n')
erros = []
amostras = []

# 1 arquivo aleatório de cada dataset
arquivos = sorted(glob.glob(os.path.join(SIGNAL_DIR, '*_signal.npz')))
por_dataset = {}
for f in arquivos:
    ds = os.path.basename(f).split('__')[0]
    por_dataset.setdefault(ds, []).append(f)

random.seed(42)
for ds, lista in sorted(por_dataset.items()):
    amostras.extend(lista)

for f in amostras:
    base = os.path.basename(f)
    z = np.load(f, allow_pickle=True)
    data   = z['data']
    sfreq  = float(z['sfreq'])
    chs    = list(z['ch_names'])
    seiz   = z['seizure_intervals']
    n_ch, n_samp = data.shape
    dur_s  = n_samp / sfreq

    ok = True

    # sfreq
    if abs(sfreq - 256) > 1:
        erros.append(f'{base}: sfreq={sfreq} (esperado 256)'); ok = False

    # canais
    if n_ch == 0:
        erros.append(f'{base}: sem canais EEG'); ok = False

    # NaN / Inf
    if not np.isfinite(data).all():
        erros.append(f'{base}: contém NaN ou Inf'); ok = False

    # sinal zerado
    if data.std() < 1e-6:
        erros.append(f'{base}: sinal zerado (filtragem falhou?)'); ok = False

    # intervalos de crise dentro da duração
    for o, e in seiz:
        if e > dur_s + 1:
            erros.append(f'{base}: crise {o:.0f}-{e:.0f}s fora da duração {dur_s:.0f}s'); ok = False

    status = 'OK' if ok else 'FALHOU'
    print(f'[{status}] {base}')
    print(f'       sfreq={sfreq:.0f} Hz | {n_ch} canais | {dur_s/60:.1f} min | '
          f'{len(seiz)} crises | std={data.std():.2f} µV')
    z.close()

print()
if erros:
    print(f'PROBLEMAS ENCONTRADOS ({len(erros)}):')
    for e in erros:
        print(f'  - {e}')
else:
    print('Todos os arquivos verificados sem problemas. Pode prosseguir para o NB2.')

=== Verificação de integridade dos _signal.npz ===

[OK] CHBMIT__chb01__chb01_01_signal.npz
       sfreq=256 Hz | 23 canais | 60.0 min | 0 crises | std=34.92 µV
[OK] CHBMIT__chb01__chb01_02_signal.npz
       sfreq=256 Hz | 23 canais | 60.0 min | 0 crises | std=30.04 µV
[OK] CHBMIT__chb01__chb01_03_signal.npz
       sfreq=256 Hz | 23 canais | 60.0 min | 1 crises | std=40.64 µV
[OK] CHBMIT__chb01__chb01_04_signal.npz
       sfreq=256 Hz | 23 canais | 60.0 min | 1 crises | std=31.54 µV
[OK] CHBMIT__chb01__chb01_05_signal.npz
       sfreq=256 Hz | 23 canais | 60.0 min | 0 crises | std=37.69 µV
[OK] CHBMIT__chb01__chb01_06_signal.npz
       sfreq=256 Hz | 23 canais | 60.0 min | 0 crises | std=37.38 µV
[OK] CHBMIT__chb01__chb01_07_signal.npz
       sfreq=256 Hz | 23 canais | 60.0 min | 0 crises | std=36.50 µV
[OK] CHBMIT__chb01__chb01_08_signal.npz
       sfreq=256 Hz | 23 canais | 60.0 min | 0 crises | std=29.56 µV
[OK] CHBMIT__chb01__chb01_09_signal.npz
       sfreq=256 Hz | 23 canais | 60

## 12. Manifesto de pacientes e arquivos

Lista explícita, por dataset e paciente, de todos os arquivos efetivamente
processados nesta execução (após exclusões da seção 2). Serve como registro
auditável de exatamente quais gravações compõem o dataset final — útil para
reprodutibilidade e para conferir rapidamente se algum paciente ficou com
menos arquivos do que o esperado.


In [17]:
manifest_rows = []
for f in sorted(glob.glob(os.path.join(SIGNAL_DIR, '*_signal.npz'))):
    base = os.path.basename(f)
    m = re.match(r'(.+?)__(.+?)__(.+)_signal\.npz', base)
    if not m:
        continue
    ds, pat, fkey = m.group(1), m.group(2), m.group(3)
    z = np.load(f, allow_pickle=True)
    manifest_rows.append(dict(
        dataset=ds, patient=pat, file=fkey,
        n_ch=int(z['data'].shape[0]),
        dur_min=round(z['data'].shape[1] / float(z['sfreq']) / 60, 1),
        has_seizure=bool(z['has_seizure']),
        n_seizures=len(z['seizure_intervals']),
    ))

df_manifest = pd.DataFrame(manifest_rows).sort_values(['dataset', 'patient', 'file'])
print(f'{len(df_manifest)} arquivos no manifesto final.\n')
print('Arquivos por paciente:')
ipd.display(df_manifest.groupby(['dataset', 'patient']).size().rename('n_arquivos'))

LOG_DIR_NB1 = os.path.join(ROOT_DIR, 'logs')
os.makedirs(LOG_DIR_NB1, exist_ok=True)
manifest_path = os.path.join(LOG_DIR_NB1, 'manifest_nb1.csv')
df_manifest.to_csv(manifest_path, index=False)
print(f'\nManifesto completo salvo em: {manifest_path}')

# Confirma que os arquivos excluídos (seção 2) de fato não aparecem
for ds, pats in EXCLUDED_FILES.items():
    for pat, files in pats.items():
        for fn in files:
            fkey = fn.replace('.edf', '')
            ainda_presente = ((df_manifest['dataset'] == ds) &
                              (df_manifest['patient'] == pat) &
                              (df_manifest['file'] == fkey)).any()
            status = 'AINDA PRESENTE (verificar!)' if ainda_presente else 'excluído com sucesso'
            print(f'  {ds}/{pat}/{fn}: {status}')


382 arquivos no manifesto final.

Arquivos por paciente:


dataset   patient
CHBMIT    chb01      15
          chb03      15
          chb04      11
          chb05      13
          chb06      15
          chb07      12
          chb08      13
          chb10      15
          chb11      11
          chb12      20
          chb13      16
          chb14      15
Mendeley  p10         2
          p11         4
          p12         3
          p13         4
          p14         3
          p15         4
SeizeIT2  sub-001     9
          sub-002    12
          sub-003     7
          sub-004     1
          sub-005     8
          sub-006     7
          sub-007     5
          sub-008     8
          sub-009     7
          sub-010     7
          sub-011     9
          sub-012     8
          sub-035    14
          sub-039     9
          sub-047     7
          sub-073    32
Siena     PN00        5
          PN01        1
          PN03        2
          PN05        3
          PN06        5
          PN07        1
          PN09        


Manifesto completo salvo em: data\logs\manifest_nb1.csv
  CHBMIT/chb12/chb12_28.edf: AINDA PRESENTE (verificar!)
  CHBMIT/chb12/chb12_29.edf: excluído com sucesso
